# Capítulo 1 — Leer los datos · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch01_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden, de arriba abajo: mientras falte una respuesta, las celdas de verificación lo indican sin producir un error. La solución está en `ch01_corrige.ipynb`, en la misma página complementaria.

**Convención.** Como en el libro, la varianza, la desviación típica, la puntuación z y la covarianza dividen por $n$ (`ddof=0`), salvo que se indique lo contrario.

**Índice.** Datos · Los cálculos de las lecciones · Ejercicios 1.1 a 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

Los identificadores (D0.1, D1.1…) son los del libro y de los archivos CSV.

Los nombres de variables y columnas del código se mantienen en francés: `logements` / `logement` = viviendas / vivienda, `surface_m2` = superficie en m², `prix_kEUR` = precio en k€, `nord` / `sud` = barrios Norte / Sur, `registre` = registro de ventas, `vente` = venta, `pieces` = habitaciones, `dpe` = clase energética, `delai_jours` = días hasta la venta, `etage` = planta, `delais_agences` = plazos de venta de las agencias, `tilleuls` / `port` = las dos residencias (Les Tilleuls, Le Port), `loyers` = alquileres, `moyenne` = media, `ecart(s)` = desviación(es).

In [ ]:
# Datos del capítulo 1, copiados de los archivos CSV del libro (no modificar)

# D0.1 — las seis viviendas de la apertura
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — precios de venta de los T2 (dos habitaciones, k€), lección 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — registro de ventas del barrio Sur, lecciones 1.2 a 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — precio por m² según la planta (k€/m²), lección 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — plazos de venta de tres agencias (días), ejercicio 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — alquileres mensuales de los estudios (€), ejercicio 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Herramientas proporcionadas: no tienes que modificarlas

def quartiles_livre(valeurs):
    """Q1 y Q3 según la regla del libro (ecuación 1.5): rangos ⌈n/4⌉ y ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # n/4 redondeado al entero superior
    rang_q3 = -(-3 * n // 4)      # 3n/4 redondeado al entero superior
    return x[rang_q1 - 1], x[rang_q3 - 1]   # rango 1 = posición 0 en Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Una fila de puntos por serie, como en las figuras de la lección 1.1.
    bandes: diccionario {nombre: (media, desviación típica)} para dibujar media ± desviación típica."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 1.1 a 1.5. Ejecútalas y compáralas con el libro. Aquí, `np.std` y `np.var` se llaman con `ddof=0`, es decir, con una división por $n$, como en el capítulo.

In [ ]:
# Lección 1.1 — media, varianza y desviación típica (división por n)
for nom, serie in [('Norte', nord), ('Sur', sud)]:
    print(f'{nom}: media = {np.mean(serie):g} k€, varianza = {np.var(serie, ddof=0):g} k€², '
          f'desviación típica = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Lección 1.2 — frecuencias por clase energética (variable cualitativa ordinal)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lección 1.3 — media, mediana y cuartiles de los plazos
d = registre['delai_jours']
print('media:', d.mean(), ' mediana:', d.median())
print('sin S13: media', d[registre['vente'] != 'S13'].mean(),
      ' mediana', d[registre['vente'] != 'S13'].median())
print('cuartiles, regla del libro (ecuación 1.5):', quartiles_livre(d))
print('cuartiles, NumPy por defecto (interpolación):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lección 1.4 — puntuaciones estandarizadas (ecuación 1.6, desviación típica con división por n)
for nom, serie, vente in [('Norte', nord, 188), ('Sur', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom}: z de la serie = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z de la venta a {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Lección 1.5 — covarianza y correlación de los cinco T2 del Sur (división por n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covarianza:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r con np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r planta / precio por m²:', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Ejercicio 1.1 — Leer un gráfico

*Datos D1.E1. Conceptos: media y desviación típica (lección 1.1).* El gráfico de abajo reproduce el del ejercicio 1.1 del libro.

In [ ]:
points_par_rangee({f'Agencia {a}': v for a, v in delais_agences.items()},
                  'Plazo de venta (días)')
plt.xlim(0, 60)
plt.show()

**a) y b)** Estima a ojo el plazo medio de cada agencia y luego ordena las agencias de la menos dispersa a la más dispersa.

In [ ]:
# PARA COMPLETAR: tus estimaciones a ojo, antes de cualquier cálculo
moyenne_estimee = {'P': None, 'Q': None, 'R': None}   # en días
ordre_dispersion = None   # lista, de la menos a la más dispersa, p. ej. ['Q', 'P', 'R']

In [ ]:
# Verificación (división por n)
for a, v in delais_agences.items():
    print(f'Agencia {a}: media = {np.mean(v):g} d, desviación típica = {np.std(v, ddof=0):.2f} d')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Completa la celda anterior.')
else:
    print('Orden de dispersión:', 'correcto' if ordre_dispersion == ordre_reel else 'revisar')

**c)** Cuenta, para Q y para R, las ventas cerradas a 2 días o menos de la media.

In [ ]:
# PARA COMPLETAR: tus recuentos, leídos en el gráfico
proches_Q = None
proches_R = None

In [ ]:
# Verificación
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agencia {a}: aún sin completar.')
    else:
        print(f"Agencia {a}: {'correcto' if la_votre == vrai else 'revisar'}")

*Tu explicación para c):* …

## Ejercicio 1.2 — Dos residencias, un mismo alquiler medio

*Datos D1.E2. Conceptos: media, varianza y desviación típica (lección 1.1).* Camille no puede pagar más de 570 € al mes. Las preguntas a) a d) se hacen primero en papel, con la tabla del ejercicio 1.2 del libro. Este notebook sirve para verificar tus respuestas, después para programar la desviación típica (pregunta e) y comparar dos convenciones (pregunta f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Alquiler mensual de un estudio (€)")
plt.axvline(570, color='black', linestyle=':', label='presupuesto de Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Leer los alquileres

¿Cuántos estudios puede alquilar Camille en cada residencia con 570 €, y luego con 620 €?

In [ ]:
# PARA COMPLETAR: número de estudios asequibles, leído en la lista de alquileres
accessibles = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Verificación: `<=` produce verdadero/falso, np.sum cuenta los «verdadero»
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'aún sin completar' if la_votre is None else ('correcto' if la_votre == vrai else 'revisar')
        print(f'Presupuesto {budget} €, {nom}: {etat}')

*Tu respuesta a a), en una frase:* …

### b) Las dos medias

Calcúlalas en papel y después anota tus resultados abajo.

In [ ]:
# PARA COMPLETAR: tus medias calculadas a mano (en €)
moyenne_tilleuls_main = None
moyenne_port_main = None

In [ ]:
# Verificación
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom}: completa la celda anterior.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom}: {'correcto' if ok else 'revisar'} (tu valor: {la_votre} €)")

### c) Varianza y desviación típica a mano (división por $n = 6$)

En papel, prepara para cada residencia la tabla de las desviaciones respecto a la media y de sus cuadrados (modelo: la tabla del cálculo paso a paso de la lección 1.1). Anota aquí los cuatro resultados.

In [ ]:
# PARA COMPLETAR: tus resultados calculados a mano (división por n)
variance_tilleuls_main = None   # en €²
variance_port_main = None       # en €²
ecart_type_tilleuls_main = None # en €
ecart_type_port_main = None     # en €

In [ ]:
# Verificación (división por n: ddof=0)
vos_valeurs = {
    'varianza Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'varianza Port': (variance_port_main, np.var(port, ddof=0)),
    'desviación típica Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'desviación típica Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom}: aún sin completar.')
    else:
        print(f"{nom}: {'correcto' if np.isclose(la_votre, attendu) else 'revisar'}")

### d) Interpretar

*Tu respuesta:* …

### e) Programar la desviación típica

Completa la función siguiendo la ecuación 1.3 del libro, con una división por $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

No uses ni `np.std` ni `np.var`: el objetivo es que escribas tú mismo cada paso.

In [ ]:
# Ejercicio 1.2 e) — PARA COMPLETAR
def ecart_type(valeurs):
    """Desviación típica, división por n (ecuación 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = None   # PARA COMPLETAR: ecuación 1.1
    ecarts = None    # PARA COMPLETAR: una desviación por valor
    variance = None  # PARA COMPLETAR: media de los cuadrados
    return None      # PARA COMPLETAR: vuelta a la unidad original

<details><summary><b>Pista</b> (haz clic si te atascas)</summary>

Con NumPy, una operación escrita sobre un array se aplica a cada valor: `x - moyenne` da de una vez las seis desviaciones, y `ecarts ** 2` sus seis cuadrados. `np.sum(...)` suma, `np.sqrt(...)` calcula la raíz cuadrada.

</details>

In [ ]:
# Verificación: ejecuta esta celda sin modificarla
if ecart_type(tilleuls) is None:
    print('La función todavía devuelve None: complétala.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'correcto' if np.isclose(obtenu, attendu) else f'revisar (esperado: {attendu})'
        print(f'{nom}: {obtenu:.2f} € → {etat}')

### f) Para profundizar: dividir por $n$ o por $n - 1$

Ejecuta la celda de abajo. Obtén 10,95 a mano dividiendo la suma de los cuadrados por $n - 1 = 5$. ¿Qué opción de pandas da la convención del capítulo?

*Tu respuesta:* …

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, por defecto (división por n − 1):', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (división por n)         :', loyers.std(ddof=0).round(2).to_dict())

## Ejercicio 1.3 — ¿Una venta sorprendente?

*Datos D1.1. Conceptos: puntuación estandarizada (lección 1.4, ecuación 1.6).* Un T2 (piso de dos habitaciones) acaba de venderse por 196 k€. Medias y desviaciones típicas: las de la lección 1.1, con división por $n$.

In [ ]:
# PARA COMPLETAR: tus cálculos hechos en papel
z_nord = None
z_sud = None
prix_sud_aussi_exceptionnel = None   # en k€, pregunta c)

In [ ]:
# Verificación (desviación típica con división por n)
attendus = {
    'z Norte': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Sur': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'precio Sur equivalente': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'aún sin completar' if la_votre is None else ('correcto' if np.isclose(la_votre, vrai) else 'revisar')
    print(f'{nom}: {etat}')

*Tu respuesta a b):* …

## Ejercicio 1.4 — ¿Van juntos superficie y precio?

*Datos D0.1 (las seis viviendas de la apertura). Conceptos: covarianza y correlación (lección 1.5, ecuaciones 1.7 y 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Superficie (m²)')
plt.ylabel('Precio de venta (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Antes de cualquier cálculo: ¿$r$ será positivo o negativo? ¿Cercano a 0, a 0,5 o a 1?

*Tu respuesta:* …

**b)** Completa las líneas de abajo. Usa una división por $n$ en todas partes.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = None     # PARA COMPLETAR: (x_i − x̄)(y_i − ȳ) para cada vivienda
covariance = None   # PARA COMPLETAR: ecuación 1.7, división por n
r = None            # PARA COMPLETAR: ecuación 1.8 (desviaciones típicas con ddof=0)

<details><summary><b>Pista</b></summary>

`x.mean()` da la media, `x - x.mean()` las seis desviaciones. El producto `*` de dos arrays multiplica término a término. `x.std(ddof=0)` da la desviación típica con división por $n$.

</details>

In [ ]:
# Verificación
if r is None or covariance is None:
    print('Completa la celda anterior.')
else:
    print('covarianza:', round(covariance, 2), 'm²·k€ →',
          'correcto' if np.isclose(covariance, 1637.5) else 'revisar (esperado: 1637,5)')
    print('r:', round(r, 4), '→', 'correcto' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'revisar')

**c)** Una correlación cercana a 1, pero desviaciones de 5 a 11 k€ en valor absoluto: ¿es contradictorio?

*Tu respuesta:* …

In [ ]:
print('np.cov por defecto (división por n − 1):', np.cov(x, y)[0, 1])
print('np.cov con bias=True (división por n)  :', np.cov(x, y, bias=True)[0, 1])

## Ejercicio 1.5 — Predecir y luego verificar

*Datos D1.1. Conceptos: media y desviación típica (lección 1.1), mediana y rango intercuartílico (lección 1.3).* Un sexto T2 se vende en el Norte por 240 k€.

**a)** Antes de cualquier cálculo, anota tus predicciones: ¿la media, la mediana, la desviación típica y el rango intercuartílico aumentan mucho, un poco o nada?

*Tus predicciones:* …

In [ ]:
nord_plus = nord + [240]
print('Norte:', sorted(nord))
print('Norte con la venta a 240 k€:', sorted(nord_plus))

**b)** Completa la función `resume`. Debe devolver los cuatro indicadores, con la desviación típica dividida por $n$ y los cuartiles calculados con la regla del libro (`quartiles_livre`, proporcionada más arriba). Las claves del diccionario se quedan en francés: `moyenne` = media, `médiane` = mediana, `écart-type` = desviación típica, `écart interquartile` = rango intercuartílico.

In [ ]:
# PARA COMPLETAR
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': None,              # PARA COMPLETAR
        'médiane': None,              # PARA COMPLETAR: np.median
        'écart-type': None,           # PARA COMPLETAR: división por n
        'écart interquartile': None,  # PARA COMPLETAR: a partir de q1 y q3
    }

In [ ]:
tableau = pd.DataFrame({'Norte (5 ventas)': resume(nord), 'Norte+240 k€ (6 ventas)': resume(nord_plus),
                        'Sur (5 ventas)': resume(sud)})
if tableau.isna().any().any():
    print('Completa la función resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k}: {'correcto' if ok else 'revisar'}")
tableau.round(2)

In [ ]:
# Las dos series según el modelo de la figura «media ± desviación típica» de la lección 1.1
if tableau.isna().any().any():
    print('Completa la función resume para mostrar el gráfico.')
else:
    points_par_rangee({'Norte': nord, 'Norte + 240': nord_plus},
                      'Precio de venta de un T2 (k€)',
                      bandes={'Norte': (np.mean(nord), np.std(nord, ddof=0)),
                              'Norte + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** ¿Se han vuelto los precios del Norte más imprevisibles que los del Sur?

*Tu respuesta:* …

## Ejercicio 1.6 — Desafío: dos rectas, un mismo error medio

*Datos D0.1. Conceptos: desviación con signo y valor absoluto (apertura, lección 1.1), desviación típica (lección 1.1).* Recta trazada a ojo: precio predicho = 3 × superficie + 30. Recta perezosa: precio predicho = 222 k€ para todas las viviendas.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = None          # PARA COMPLETAR: observado − predicho, recta trazada a ojo
ecarts_paresseuse = None    # PARA COMPLETAR: observado − predicho, recta perezosa

In [ ]:
# Los tres indicadores de la pregunta b) (desviación típica: división por n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Completa la celda anterior.')
else:
    indicateurs = pd.DataFrame({
        nom: {'media de las desviaciones con signo': e.mean(),
              'media de las desviaciones en valor absoluto': np.abs(e).mean(),
              'desviación típica de las desviaciones con signo': e.std(ddof=0)}
        for nom, e in [('recta a ojo', ecarts_oeil), ('recta perezosa', ecarts_paresseuse)]
    })
    print('desviaciones, recta a ojo    :', ecarts_oeil)
    print('desviaciones, recta perezosa :', ecarts_paresseuse)
    print('verificación:', 'correcto' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'revisar')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="recta a ojo")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='recta perezosa')
    ax.set_xlabel('Superficie (m²)')
    ax.set_ylabel('Desviación observado − predicho (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Tus respuestas a c) y d):* …

In [ ]:
# Control de d): la desviación típica de las desviaciones de la recta perezosa es igual a la de los precios
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))